# 01 — Data Generation

Generates a realistic, internally-consistent synthetic ride-sharing dataset
(Riders, Drivers, Vehicles, Locations, Trips, Payments, Ratings) and writes
it to `data/raw/*.csv`, matching the schema in `sql/02_create_tables.sql`.

The generation model deliberately bakes in real-world signal — surge
pricing raising cancellation odds, late-night trips being riskier, newer
drivers cancelling more, etc. — so that later EDA, statistical tests and
ML have real patterns to recover, not noise.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from faker import Faker

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
fake = Faker()
Faker.seed(RANDOM_SEED)

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

N_RIDERS = 3000
N_DRIVERS = 450
N_TRIPS = 45000

CITIES = {
    "Delhi":     (28.6139, 77.2090),
    "Mumbai":    (19.0760, 72.8777),
    "Bangalore": (12.9716, 77.5946),
    "Chennai":   (13.0827, 80.2707),
    "Hyderabad": (17.3850, 78.4867),
    "Pune":      (18.5204, 73.8567),
}
VEHICLE_TYPES = ["Bike", "Auto", "Economy", "Premium", "XL"]
VEHICLE_BASE_RATE = {"Bike": 6, "Auto": 9, "Economy": 12, "Premium": 20, "XL": 25}   # per-km rate
VEHICLE_BASE_FARE = {"Bike": 15, "Auto": 25, "Economy": 40, "Premium": 80, "XL": 100}
PAYMENT_METHODS = ["Card", "Cash", "Wallet", "UPI"]
CANCELLATION_REASONS_RIDER = [
    "Found alternate ride", "Driver taking too long", "Changed plans",
    "Price too high", "Booked by mistake",
]
CANCELLATION_REASONS_DRIVER = [
    "Rider not reachable", "Vehicle breakdown", "Too far from pickup",
    "Unsafe pickup location", "Personal emergency",
]

TRIP_START = datetime(2025, 1, 1)
TRIP_END = datetime(2025, 10, 31, 23, 59)

## Riders

In [2]:
rider_ids = np.arange(1, N_RIDERS + 1)
rider_signup = [
    fake.date_between(start_date=TRIP_START - timedelta(days=365), end_date=TRIP_END)
    for _ in range(N_RIDERS)
]
riders = pd.DataFrame({
    "rider_id": rider_ids,
    "first_name": [fake.first_name() for _ in range(N_RIDERS)],
    "last_name": [fake.last_name() for _ in range(N_RIDERS)],
    "email": [f"rider{i}@example.com" for i in rider_ids],
    "phone": [fake.numerify("9#########") for _ in range(N_RIDERS)],
    "gender": np.random.choice(["Male", "Female", "Other"], N_RIDERS, p=[0.52, 0.46, 0.02]),
    "age": np.random.randint(18, 65, N_RIDERS),
    "city": np.random.choice(list(CITIES.keys()), N_RIDERS),
    "signup_date": rider_signup,
    "preferred_payment": np.random.choice(PAYMENT_METHODS, N_RIDERS, p=[0.30, 0.15, 0.20, 0.35]),
})

## Drivers + Vehicles (1-1)

In [3]:
driver_ids = np.arange(1, N_DRIVERS + 1)
driver_signup = [
    fake.date_between(start_date=TRIP_START - timedelta(days=730), end_date=TRIP_END)
    for _ in range(N_DRIVERS)
]
drivers = pd.DataFrame({
    "driver_id": driver_ids,
    "first_name": [fake.first_name() for _ in range(N_DRIVERS)],
    "last_name": [fake.last_name() for _ in range(N_DRIVERS)],
    "email": [f"driver{i}@example.com" for i in driver_ids],
    "phone": [fake.numerify("9#########") for _ in range(N_DRIVERS)],
    "gender": np.random.choice(["Male", "Female", "Other"], N_DRIVERS, p=[0.86, 0.13, 0.01]),
    "city": np.random.choice(list(CITIES.keys()), N_DRIVERS),
    "license_number": [fake.unique.bothify("??######").upper() for _ in range(N_DRIVERS)],
    "signup_date": driver_signup,
    "driver_status": np.random.choice(["Active", "Inactive", "Suspended"], N_DRIVERS, p=[0.88, 0.09, 0.03]),
    "avg_rating": np.round(np.clip(np.random.normal(4.3, 0.4, N_DRIVERS), 1.0, 5.0), 2),
})

vehicles = pd.DataFrame({
    "vehicle_id": driver_ids,
    "driver_id": driver_ids,
    "vehicle_type": np.random.choice(VEHICLE_TYPES, N_DRIVERS, p=[0.20, 0.20, 0.35, 0.15, 0.10]),
    "make": np.random.choice(["Maruti", "Hyundai", "Tata", "Honda", "Toyota", "Bajaj", "Hero"], N_DRIVERS),
    "model": [fake.word().capitalize() for _ in range(N_DRIVERS)],
    "year": np.random.randint(2012, 2025, N_DRIVERS),
    "plate_number": [fake.unique.bothify("??-##-??-####").upper() for _ in range(N_DRIVERS)],
    "seating_capacity": np.random.choice([1, 3, 4, 6], N_DRIVERS, p=[0.20, 0.20, 0.45, 0.15]),
})

## Locations — a handful of areas per city with jittered lat/long

In [4]:
location_rows = []
loc_id = 1
for city, (lat, lon) in CITIES.items():
    for _ in range(8):
        location_rows.append({
            "location_id": loc_id,
            "city": city,
            "area_name": fake.street_name(),
            "latitude": round(lat + np.random.uniform(-0.15, 0.15), 6),
            "longitude": round(lon + np.random.uniform(-0.15, 0.15), 6),
        })
        loc_id += 1
locations = pd.DataFrame(location_rows)

## Trips

Each trip's cancellation probability is a logistic function of surge
multiplier, hour of day, driver rating, and trip distance — this is what
gives the statistical tests and ML models real signal to find later.

In [5]:
def random_datetime(start, end, n):
    delta_seconds = int((end - start).total_seconds())
    offsets = np.random.randint(0, delta_seconds, n)
    # weight toward commute hours (7-10, 17-21) using rejection-free hour remap
    return [start + timedelta(seconds=int(s)) for s in offsets]

n = N_TRIPS
trip_ids = np.arange(1, n + 1)

# riders/drivers weighted so some are far more active than others (power-law-ish)
rider_weights = np.random.pareto(1.5, N_RIDERS) + 0.1
rider_weights /= rider_weights.sum()
trip_rider_ids = np.random.choice(rider_ids, n, p=rider_weights)

active_drivers = drivers.loc[drivers.driver_status == "Active", "driver_id"].values
driver_weights = np.random.pareto(1.3, len(active_drivers)) + 0.1
driver_weights /= driver_weights.sum()
trip_driver_ids = np.random.choice(active_drivers, n, p=driver_weights)

trip_vehicle_ids = trip_driver_ids  # 1-1 with driver
vehicle_type_lookup = vehicles.set_index("vehicle_id")["vehicle_type"]
trip_vehicle_types = vehicle_type_lookup.loc[trip_vehicle_ids].values

driver_rating_lookup = drivers.set_index("driver_id")["avg_rating"]
trip_driver_ratings = driver_rating_lookup.loc[trip_driver_ids].values

driver_signup_lookup = drivers.set_index("driver_id")["signup_date"]
trip_driver_signup = pd.to_datetime(driver_signup_lookup.loc[trip_driver_ids].values)

# request time: biased toward commute peaks via mixture of normals on hour-of-day
day_offsets = np.random.randint(0, (TRIP_END - TRIP_START).days, n)
base_dates = np.array([TRIP_START + timedelta(days=int(d)) for d in day_offsets])
hour_component = np.random.choice(
    np.arange(24), n,
    p=(lambda h: (h / h.sum()))(
        np.array([
            1, 1, 1, 1, 1, 2, 4, 8, 10, 7, 5, 5,
            6, 5, 5, 6, 7, 9, 10, 8, 6, 4, 3, 2,
        ], dtype=float)
    ),
)
minute_component = np.random.randint(0, 60, n)
request_datetime = np.array([
    base_dates[i].replace(hour=int(hour_component[i]), minute=int(minute_component[i]))
    for i in range(n)
])

rider_city_lookup = riders.set_index("rider_id")["city"]
trip_cities = rider_city_lookup.loc[trip_rider_ids].values

loc_by_city = {c: locations.loc[locations.city == c, "location_id"].values for c in CITIES}
pickup_location_id = np.array([np.random.choice(loc_by_city[c]) for c in trip_cities])
drop_location_id = np.array([
    np.random.choice(loc_by_city[c][loc_by_city[c] != p]) if len(loc_by_city[c]) > 1 else p
    for c, p in zip(trip_cities, pickup_location_id)
])

distance_km = np.round(np.clip(np.random.gamma(shape=2.2, scale=2.8, size=n), 0.8, 45), 2)
duration_min = np.round(distance_km * np.random.uniform(2.3, 3.6, n) + np.random.normal(0, 2, n), 1)
duration_min = np.clip(duration_min, 3, None)

# surge multiplier driven by hour-of-day + weekend + a little noise
is_weekend = pd.Series(request_datetime).dt.dayofweek.isin([5, 6]).values.astype(int)
hour_arr = pd.Series(request_datetime).dt.hour.values
peak_hour = np.isin(hour_arr, [8, 9, 18, 19, 20]).astype(int)
surge_base = 1.0 + 0.35 * peak_hour + 0.15 * is_weekend
surge_multiplier = np.round(np.clip(surge_base + np.random.normal(0, 0.15, n), 1.0, 3.0), 2)

vehicle_base_fare = np.array([VEHICLE_BASE_FARE[v] for v in trip_vehicle_types])
vehicle_rate = np.array([VEHICLE_BASE_RATE[v] for v in trip_vehicle_types])
base_fare = np.round(vehicle_base_fare, 2)
computed_fare = np.round((base_fare + vehicle_rate * distance_km) * surge_multiplier + np.random.normal(0, 5, n), 2)
computed_fare = np.clip(computed_fare, 10, None)

# --- cancellation probability model ---
driver_experience_days = (pd.Series(request_datetime) - pd.Series(trip_driver_signup)).dt.days.values
driver_experience_days = np.clip(driver_experience_days, 0, None)

z = (
    -2.3
    + 2.3 * (surge_multiplier - 1.0)
    + 1.0 * ((hour_arr >= 23) | (hour_arr <= 4)).astype(int)
    + 1.0 * (driver_rating_lookup.loc[trip_driver_ids].values < 3.8).astype(int)
    + 0.7 * (driver_experience_days < 30).astype(int)
    + 0.03 * distance_km
    - 0.15 * is_weekend
    + np.random.normal(0, 0.45, n)
)
cancel_prob = 1 / (1 + np.exp(-z))
is_cancelled = np.random.binomial(1, cancel_prob)

# split cancellations into rider-cancelled / driver-cancelled / no-driver-found
cancel_type_roll = np.random.rand(n)
trip_status = np.empty(n, dtype=object)
cancellation_reason = np.full(n, None, dtype=object)

for i in range(n):
    if is_cancelled[i] == 0:
        trip_status[i] = "Completed"
    else:
        r = cancel_type_roll[i]
        if r < 0.55:
            trip_status[i] = "Cancelled_by_Rider"
            cancellation_reason[i] = np.random.choice(CANCELLATION_REASONS_RIDER)
        elif r < 0.90:
            trip_status[i] = "Cancelled_by_Driver"
            cancellation_reason[i] = np.random.choice(CANCELLATION_REASONS_DRIVER)
        else:
            trip_status[i] = "No_Driver_Found"
            cancellation_reason[i] = "No driver accepted the request"

completed_mask = trip_status == "Completed"

# Only completed trips ever actually start (get picked up) or end (get dropped) —
# cancelled/no-driver-found trips never have a real pickup or drop time.
request_dt_series = pd.Series(request_datetime)
pickup_offset = pd.to_timedelta(np.random.randint(2, 12, n), unit="m")

pickup_datetime = pd.Series(pd.NaT, index=range(n), dtype="datetime64[ns]")
pickup_datetime.loc[completed_mask] = (request_dt_series + pickup_offset).loc[completed_mask]

drop_datetime = pd.Series(pd.NaT, index=range(n), dtype="datetime64[ns]")
drop_datetime.loc[completed_mask] = (
    pickup_datetime.loc[completed_mask] + pd.to_timedelta(duration_min[completed_mask], unit="m")
)

pickup_datetime = pickup_datetime.values
drop_datetime = drop_datetime.values

fare_amount = np.where(completed_mask, computed_fare, np.nan)
final_distance = np.where(trip_status == "No_Driver_Found", 0.0, distance_km)
final_duration = np.where(completed_mask, duration_min, np.nan)

payment_method = np.random.choice(PAYMENT_METHODS, n, p=[0.30, 0.15, 0.20, 0.35])

trips = pd.DataFrame({
    "trip_id": trip_ids,
    "rider_id": trip_rider_ids,
    "driver_id": trip_driver_ids,
    "vehicle_id": trip_vehicle_ids,
    "pickup_location_id": pickup_location_id,
    "drop_location_id": drop_location_id,
    "request_datetime": request_datetime,
    "pickup_datetime": pickup_datetime,
    "drop_datetime": drop_datetime,
    "distance_km": final_distance,
    "duration_min": final_duration,
    "base_fare": base_fare,
    "surge_multiplier": surge_multiplier,
    "fare_amount": fare_amount,
    "trip_status": trip_status,
    "cancellation_reason": cancellation_reason,
    "payment_method": payment_method,
}).sort_values("request_datetime").reset_index(drop=True)
trips["trip_id"] = np.arange(1, n + 1)  # re-sequence ids in chronological order

print(trips.trip_status.value_counts(normalize=True).round(3))

trip_status
Completed              0.789
Cancelled_by_Rider     0.116
Cancelled_by_Driver    0.074
No_Driver_Found        0.021
Name: proportion, dtype: float64


## Payments

In [6]:
payment_status = np.where(
    trips.trip_status == "Completed",
    np.random.choice(["Success", "Failed", "Refunded"], n, p=[0.965, 0.015, 0.02]),
    "Not_Applicable",
)
payments = pd.DataFrame({
    "payment_id": trips.trip_id,
    "trip_id": trips.trip_id,
    "amount": trips.fare_amount.fillna(0).round(2),
    "payment_method": trips.payment_method,
    "payment_status": payment_status,
    "payment_datetime": trips.drop_datetime,
})

## Ratings — only for completed trips, with some missingness

In [7]:
completed_trips = trips.loc[trips.trip_status == "Completed"].copy()
has_rating = np.random.rand(len(completed_trips)) < 0.82

rider_rating_raw = np.round(np.clip(np.random.normal(4.4, 0.7, len(completed_trips)), 1, 5))
driver_rating_raw = np.round(np.clip(np.random.normal(4.5, 0.6, len(completed_trips)), 1, 5))
rider_rating_for_driver = pd.Series(pd.array(rider_rating_raw, dtype="Int64")).where(has_rating, pd.NA)
driver_rating_for_rider = pd.Series(pd.array(driver_rating_raw, dtype="Int64")).where(has_rating, pd.NA)

feedback_pool = [
    "Great ride, on time!", "Driver was very polite", "Car could be cleaner",
    "Smooth trip", None, None, None, "Took a longer route than expected",
    "Excellent service", "A bit late to pickup",
]
feedback_text = np.random.choice(feedback_pool, len(completed_trips))

ratings = pd.DataFrame({
    "rating_id": completed_trips.trip_id,
    "trip_id": completed_trips.trip_id,
    "rider_rating_for_driver": rider_rating_for_driver.values,
    "driver_rating_for_rider": driver_rating_for_rider.values,
    "feedback_text": feedback_text,
})

## Inject realistic data-quality issues

Real operational data is never clean. We deliberately introduce a small,
controlled amount of missingness/duplication so `02_data_cleaning.ipynb`
has real work to do.

In [8]:
dup_rows = trips.sample(25, random_state=RANDOM_SEED)
trips_with_dupes = pd.concat([trips, dup_rows], ignore_index=True)

missing_idx = trips_with_dupes.sample(frac=0.01, random_state=RANDOM_SEED).index
trips_with_dupes.loc[missing_idx, "duration_min"] = np.nan

trips_final = trips_with_dupes

## Write raw CSVs

In [9]:
riders.to_csv(RAW_DIR / "riders.csv", index=False)
drivers.to_csv(RAW_DIR / "drivers.csv", index=False)
vehicles.to_csv(RAW_DIR / "vehicles.csv", index=False)
locations.to_csv(RAW_DIR / "locations.csv", index=False)
trips_final.to_csv(RAW_DIR / "trips.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)
ratings.to_csv(RAW_DIR / "ratings.csv", index=False)

print("Rows written:")
for name, df in [
    ("riders", riders), ("drivers", drivers), ("vehicles", vehicles),
    ("locations", locations), ("trips", trips_final),
    ("payments", payments), ("ratings", ratings),
]:
    print(f"  {name:10s} {len(df):>7,}")

Rows written:
  riders       3,000
  drivers        450
  vehicles       450
  locations       48
  trips       45,025
  payments    45,000
  ratings     35,507
